In [47]:
import os
import json
from openai import OpenAI


class DataGenerator:
    def __init__(self):
        self.api_key = os.environ["OPENAI_RESEARCH"]
        self.client = OpenAI(api_key=self.api_key)

        self.current_batch = None

    def process_batch_request(self, batch_requests_dir: str):
        batch_input_file = self.client.files.create(
            file=open(batch_requests_dir, "rb"), purpose="batch"
        )

        batch_object = self.client.batches.create(
            input_file_id=batch_input_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
        )

        self.current_batch = batch_object

        print(f"Created batch request. ID: {batch_object.id}")

    def save_batch_response(self, output_dir: str):

        self.current_batch = self.client.batches.retrieve(self.current_batch.id)

        if self.batch_completed():
            file_response = self.client.files.content(self.current_batch.output_file_id)

            jsonl_lines = file_response.text.strip().split("\n")

            with open(output_dir, "a") as outfile:
                for line in jsonl_lines:
                    json_object = json.loads(line)
                    response_message = json_object["response"]["body"]["choices"][0][
                        "message"
                    ]["content"]
                    outfile.write(response_message + "\n")

            self.current_batch = None

        else:
            print(
                f"Batch not completed yet. Status: {self.current_batch.status}   Progress: {self.current_batch.request_counts}"
            )

    def batch_completed(self):
        status = self.current_batch.status
        if status == "completed":
            return True
        return False

In [48]:
data_generator = DataGenerator()

In [49]:
data_generator.process_batch_request("../data/requests/bh_requests.jsonl")

Created batch request. ID: batch_679b8606eee8819080f27d082e1a2dff


In [53]:
data_generator.save_batch_response("../data/unlabeled/bh_texts.txt")

Batch not completed yet. Status: failed   Progress: BatchRequestCounts(completed=0, failed=0, total=0)


In [54]:
data_generator.current_batch

Batch(id='batch_679b8606eee8819080f27d082e1a2dff', completion_window='24h', created_at=1738245639, endpoint='/v1/chat/completions', input_file_id='file-RCx5UAEBcNurcYBTf1q613', object='batch', status='failed', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=Errors(data=[BatchError(code='token_limit_exceeded', line=None, message='Enqueued token limit reached for gpt-4o in organization org-MZ1JgZsueKJwV5gBmoTXKqZ2. Limit: 90,000 enqueued tokens. Please try again once some in_progress batches have been completed.', param=None)], object='list'), expired_at=None, expires_at=1738332039, failed_at=1738245640, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))